# 8. Hybrid + 0.55 cutoff + Gemini

Large precision and negative-abstention gain, but severe false answerable abstention.

This is a recorded full-train comparison, not a new evaluation.

In [1]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0,str(ROOT/'src'))
sys.path.insert(0,str(ROOT/'scripts'))
from presentation_review import selection,cached_demo
model=selection(8)
model['config_snapshot']

{'name': 'llm_hybrid_gemini_summary_055',
 'provider': 'gemini',
 'task': 'hybrid_summary_with_evidence',
 'model': 'gemini-3.5-flash-lite',
 'model_revision': 'API alias; actual modelVersion recorded per response',
 'runtime': 'gemini-rest-v1beta',
 'prompt_file': 'prompts/summarise_with_evidence_v1.txt',
 'temperature': 0,
 'seed': None,
 'max_output_tokens': 1536,
 'context_length': 1048576,
 'thinking_level': 'minimal',
 'candidate_k': 5,
 'retrieval_config': 'configs/hybrid_bm25_minilm_rrf_abstention.json',
 'max_evidence_quotes': 3,
 'max_summary_words': 45,
 'invalid_output_policy': 'return_original_candidates',
 'expose_in_demo': True}

In [2]:
m=model['train_metrics']
table=pd.DataFrame(m['retrieval'])[['k','n_queries','precision','recall','hit']]
table

,k,n_queries,precision,recall,hit
0,1,140,0.555114,0.386910,0.578571
1,3,140,0.541879,0.466243,0.607143
2,5,140,0.541523,0.469310,0.607143
3,10,140,0.541523,0.469310,0.607143


In [3]:
print('Correct negative abstention:',m['negative_counts'])
print('False answerable abstention:',m['false_abstention'])
print('Cached latency:',m['latency_ms'])

Correct negative abstention: {'n': 23, 'correct': 23}
False answerable abstention: 0.32857142857142857
Cached latency: {'median': 11.522900022100657, 'p95': 14.880775019992138}


## Cached example

The question comes from recorded train predictions. Config settings and original source offsets remain visible; the model cannot call a hosted provider here.

In [4]:
demo=cached_demo(model)
print(demo['query'])
print(demo.get('summary') or demo['message'])
pd.DataFrame(demo['hits'])[['file_path','start','end','text']]

Consider the Amendment No. 3 to Wireless Content License Agreement between Fox Mobile Entertainment, Inc. and Glu Mobile Inc.; What is the expiration date of this contract?
The contract term expires on December 31, 2006, but rights and licenses for specific properties continue until March 31, 2008.


,file_path,start,end,text
0,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,592,988,TERM: The rights granted hereunder shall be ef...


Full code walkthrough: ../experiments/19_hybrid_gemini_summary.ipynb. Historical acceptance and current preference are separate; see REPORT.md for limitations.